In [ ]:
import pyodbc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Connect to SQL Server
conn_str = (
    "DRIVER={SQL Server};"
    "SERVER=YourServerName;"
    "DATABASE=YourDatabaseName;"
    "Trusted_Connection=yes;"  
    
    "UID=YourUsername;"
    "PWD=YourPassword;"
)

try:
    conn = pyodbc.connect(conn_str)
    print("Connected to SQL Server successfully!")

In [ ]:
    # 1. Sales by Customer for All Time (Horizontal Bar Chart)
    # -------------------------------------------------------
    sales_by_customer_query = """
    SELECT 
        c.CustomerID,
        c.Name AS CustomerName,
        SUM(od.Quantity * od.Price) AS TotalSales
    FROM 
        Customer c
    JOIN 
        OrderHeader oh ON c.CustomerID = oh.CustomerID
    JOIN 
        OrderDetail od ON oh.OrderNumber = od.OrderNumber
    GROUP BY 
        c.CustomerID, c.Name
    ORDER BY 
        TotalSales DESC
    """
    
    sales_by_customer_df = pd.read_sql(sales_by_customer_query, conn)
    
    plt.figure(figsize=(12, 10))
    bar_plot = sns.barplot(
        x='TotalSales', 
        y='CustomerName', 
        data=sales_by_customer_df,
        palette='viridis'
    )
    
    plt.title('Total Sales by Customer', fontsize=16)
    plt.xlabel('Total Sales (USD)', fontsize=12)
    plt.ylabel('Customer', fontsize=12)
    
    for i, v in enumerate(sales_by_customer_df['TotalSales']):
        bar_plot.text(v + 10, i, f'${v:.2f}', va='center')
    
    plt.tight_layout()
    plt.savefig('sales_by_customer.png', dpi=300)
    plt.close()
    
    print("Sales by Customer chart created successfully!")
    


In [ ]:
    # 2. Sales by State for All Time (Choropleth Map)
    # ----------------------------------------------
    sales_by_state_query = """
    SELECT 
        c.State,
        SUM(od.Quantity * od.Price) AS TotalSales
    FROM 
        Customer c
    JOIN 
        OrderHeader oh ON c.CustomerID = oh.CustomerID
    JOIN 
        OrderDetail od ON oh.OrderNumber = od.OrderNumber
    GROUP BY 
        c.State
    ORDER BY 
        TotalSales DESC
    """
    
    sales_by_state_df = pd.read_sql(sales_by_state_query, conn)
    
    plt.figure(figsize=(14, 10))
    
    plt.pie(
        sales_by_state_df['TotalSales'],
        labels=sales_by_state_df['State'],
        autopct='%1.1f%%',
        startangle=90,
        shadow=True,
        explode=[0.05 if i < 3 else 0 for i in range(len(sales_by_state_df))],  # Explode the top 3 states
        textprops={'fontsize': 12}
    )

    plt.title('Sales Distribution by State', fontsize=16)
    plt.axis('equal')
    
    plt.tight_layout()
    plt.savefig('sales_by_state_pie.png', dpi=300)
    plt.close()
    
    plt.figure(figsize=(14, 8))
    bar_plot = sns.barplot(
        x='State', 
        y='TotalSales', 
        data=sales_by_state_df,
        palette='coolwarm'
    )
    
    plt.title('Total Sales by State', fontsize=16)
    plt.xlabel('State', fontsize=12)
    plt.ylabel('Total Sales (USD)', fontsize=12)
    plt.xticks(rotation=45)
    
    for i, v in enumerate(sales_by_state_df['TotalSales']):
        bar_plot.text(i, v + 10, f'${v:.2f}', ha='center')
 
    plt.tight_layout()
    plt.savefig('sales_by_state_bar.png', dpi=300)
    plt.close()
    
    print("Sales by State charts created successfully!")
    


In [ ]:
    # Additional visualization: Top selling products
    # ----------------------------------------------
    top_products_query = """
    SELECT 
        od.ItemName,
        SUM(od.Quantity * od.Price) AS TotalSales
    FROM 
        OrderDetail od
    GROUP BY 
        od.ItemName
    ORDER BY 
        TotalSales DESC
    """
    
    top_products_df = pd.read_sql(top_products_query, conn)
    
    top_10_products = top_products_df.head(10)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(
        x='TotalSales', 
        y='ItemName', 
        data=top_10_products,
        palette='magma'
    )
    
    plt.title('Top 10 Products by Sales', fontsize=16)
    plt.xlabel('Total Sales (USD)', fontsize=12)
    plt.ylabel('Product', fontsize=12)
    
    plt.tight_layout()
    plt.savefig('top_products.png', dpi=300)
    plt.close()
    
    print("Top Products chart created successfully!")
    
    print("All visualizations completed successfully!")
    
except Exception as e:
    print(f"Error: {str(e)}")
    
finally:
    if 'conn' in locals():
        conn.close()
        print("Connection closed.")